# 05 - Initial research questions and the bridge to preference estimation and bundle design

This notebook closes the descriptive EDA stage and records the corrected bridge to the live two-stage project. It does two things:

1. Describes the observed relationship between an overlap-adjusted ownership-attribution proxy, discount, size, and component price.
2. Explains why that reduced-form association cannot identify a pricing response, then links leakage-safe latent-score estimation to the primary CP-anchored fixed-bundle $\mathrm{SBA}^{CP}$ experiment through declared pseudo-utility scenarios.

Notebook 05 was originally written before notebook 04. This corrected version uses `mincost_attributed_count` from notebook 04, with `users_own_all` and `users_own_any` retained only as alternative descriptive proxies.

Candidate questions the available data and declared model can support are:

- **A. Descriptive bundle evidence.** How is the ownership-attribution proxy associated with observed discount, size, and component price?
- **B. Preference-score estimation.** How accurately do the required implicit-feedback models reconstruct leakage-safe held-out ownership under full-catalogue ranking?
- **C. Conditional decision comparison.** Under frozen pseudo-utility scenarios, how do normalized objectives for CP-anchored SBA and distinct benchmarks vary across models, transformations, candidate pools, capacities, costs, and tie rules?
- **D. Fixed-bundle design.** Which feasible seller-curated composition and normalized bundle price maximize the stated finite-panel objective, and when can that claim be certified?

The live line is **A, then B, then D with C as robustness and benchmarking**. The descriptive regression is not a demand model. Latent scores remain ranking objects, pseudo-utilities remain modeling scenarios, and all decision results remain conditional normalized comparisons rather than estimates of willingness to pay, purchase probabilities, or Steam revenue.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

## Section 1: Load and merge

The demand source is notebook 04's `bundle_mincost_attribution.csv`, which already carries the bundle metadata (name, size, prices, implied discount), `panel_coverage`, the raw `users_own_all`, and the overlap-adjusted `mincost_attributed_count`. The only field it lacks is `users_own_any`, which we pull from notebook 03's `bundle_demand_proxy.csv` for the robustness comparison.

We restrict to full-coverage bundles with at least one owner (`panel_coverage == 1.0` and `users_own_any > 0`), the same trust rule used in notebooks 03 and 04: outside full coverage, owning-all counts are mechanically biased toward zero.

In [2]:
att = pd.read_csv('../outputs/tables/bundle_mincost_attribution.csv', dtype={'bundle_id': str})
proxy = pd.read_csv('../outputs/tables/bundle_demand_proxy.csv', dtype={'bundle_id': str})

df = att.merge(proxy[['bundle_id', 'n_bundle_items', 'users_own_any', 'avg_playtime_overlap']], on='bundle_id', how='left')
# full-coverage bundles with at least one owner in the panel
d = df[(df['panel_coverage'] == 1.0) & (df['users_own_any'] > 0)].copy() # >0 to avoid log(0) issues later
print(f'usable full-coverage bundles: {len(d)}')

# mincost attributed count should not create users; (<= user owns all)
assert (d['mincost_attributed_count'] <= d['users_own_all'] + 1e-6).all()

d['demand'] = d['mincost_attributed_count'] # approximate demand
d['log_demand'] = np.log1p(d['demand']) # 1p to handle zero demand bundles; log(1 + x)
d['log_own_all'] = np.log1p(d['users_own_all'])
d['log_own_any'] = np.log(d['users_own_any'])
d['log_price'] = np.log(d['component_price_sum'])
d['is_zero_discount'] = (d['implied_discount_rate'] == 0).astype(int) # 1 if zero discount, 0 otherwise

print(f"zero-discount bundles in this set: {int(d['is_zero_discount'].sum())}")
d[['mincost_attributed_count', 'users_own_all', 'users_own_any',
   'implied_discount_rate', 'n_bundle_items', 'component_price_sum']].describe().round(2)

usable full-coverage bundles: 238
zero-discount bundles in this set: 4


,mincost_attributed_count,users_own_all,users_own_any,implied_discount_rate,n_bundle_items,component_price_sum
count,238.00,238.00,238.00,238.00,238.00,238.00
mean,177.47,193.79,1381.68,0.29,4.13,34.50
std,1099.27,1164.21,4026.66,0.19,3.59,38.77
min,0.00,0.00,1.00,0.00,2.00,1.98
25%,1.00,2.00,50.75,0.15,2.00,10.58
50%,6.00,7.00,195.00,0.25,3.00,22.97
75%,22.00,26.75,956.00,0.35,5.00,39.97
max,13837.00,13837.00,37299.00,0.92,33.00,277.67


## Section 2: How notebook 04 changed the proxy

Before regressing, it is worth seeing why the proxy switch matters rather than asserting it. Notebook 04 resolved overlap (a user who owns the union of two competing bundles should not be counted fully toward both) and also split the credit of zero-discount bundles (where buying the bundle and buying its games solo cost the same). Both effects move `mincost_attributed_count` below `users_own_all`.

The table below shows the full-coverage bundles where the two proxies disagree most. The biggest single change is a zero-discount bundle (BioShock Triple Pack, 6951 to 3475.5), which is exactly the case notebook 04 said to flag rather than silently halve. We handle that in the regression with a zero-discount indicator (Section 3).

In [3]:
d['proxy_change'] = d['users_own_all'] - d['mincost_attributed_count']
n_changed = int((d['proxy_change'].abs() > 0.5).sum()) # Changed by more than 0.5 users
print(f'full-coverage bundles where the two proxies disagree: {n_changed} of {len(d)}')

changed = (d[d['proxy_change'].abs() > 0.5]
           .sort_values(['proxy_change', 'bundle_id'], ascending=[False, True], kind='mergesort')
           [['bundle_name', 'implied_discount_rate', 'contested_candidate_share', 'users_own_all', 'mincost_attributed_count']]
           .rename(columns={'implied_discount_rate': 'discount', 'contested_candidate_share': 'contested_share'}))
print(changed.head(8).round(3).to_string(index=False))

full-coverage bundles where the two proxies disagree: 49 of 238
                           bundle_name  discount  contested_share  users_own_all  mincost_attributed_count
                  BioShock Triple Pack     0.000            0.000           6951                    3475.5
                   Interstellar Bundle     0.568            1.000             79                       5.0
              Stronghold Complete Pack     0.000            0.000             65                      32.5
Platformer Bundle by New Reality Games     0.504            1.000             27                       0.0
                          Space Bundle     0.391            0.522            159                     137.0
                             Trinelogy     0.300            0.055            385                     364.0
           Pilgrim Adventures Complete     0.352            1.000             17                       0.0
          The Indie Platformers Bundle     0.860            0.516             31

## Section 3: Does demand vary with discount, size, and price?

Regress log demand (the overlap-adjusted proxy) on the implied discount rate, bundle size, log component price, and a zero-discount indicator. This is **descriptive, not causal**: publishers choose discounts strategically (often discounting weaker bundles more), so the discount is not randomly assigned and the coefficient is a conditional correlation, not a treatment effect.

Two deliberate choices:

- **Robust (HC3) standard errors.** The residuals in the original notebook were non-normal (Jarque-Bera flagged) and heteroskedasticity is likely across bundles spanning three orders of magnitude in demand, so HC3 is the honest default over classical standard errors.
- **A zero-discount indicator.** The four zero-discount full-coverage bundles are the ones whose attributed count notebook 04 splits with the all-solo path. The indicator records this mechanically different attribution case and keeps all 238 bundles in the fit. Conditioning on it does not remove the broader confounding in the discount coefficient.

In [4]:
model = smf.ols('log_demand ~ implied_discount_rate + n_bundle_items + log_price + is_zero_discount',data=d
                ).fit(cov_type='HC3') # HC3 allows residual variance to vary across observatiions, regualating variance
print(model.summary())
assert int(model.nobs) == len(d)

                            OLS Regression Results                            
Dep. Variable:             log_demand   R-squared:                       0.079
Model:                            OLS   Adj. R-squared:                  0.063
Method:                 Least Squares   F-statistic:                     5.370
Date:                Fri, 17 Jul 2026   Prob (F-statistic):           0.000374
Time:                        17:00:24   Log-Likelihood:                -471.98
No. Observations:                 238   AIC:                             954.0
Df Residuals:                     233   BIC:                             971.3
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.84

## Interpreting the fitted specification

**1. The sample $R^2$ is 0.079.** The four regressors account for 7.9% of the observed sample variation in `log_demand`; this does not imply that a particular list of omitted variables collectively explains the remaining 92.1%. The low fit reinforces the limited descriptive role of this specification, but it is not by itself a decomposition of demand and does not support bundle pricing.

**2. The discount coefficient is likely confounded.** Publishers choose discounts and timing. Product age, expected demand, promotional strategy, catalogue composition, and other unobserved factors can affect both the chosen discount and the ownership proxy. The fitted coefficient is therefore a conditional association, not an identified response to changing a discount.

**3. The negative bundle-size coefficient is not evidence that customers dislike larger bundles.** The ownership-attribution outcome is mechanically harder to satisfy as bundle size grows, and bundle size is also associated with catalogue composition and seller decisions. This cross-sectional coefficient establishes neither a monotone preference for smaller bundles nor a causal size effect.

**4. The zero-discount coefficient is not statistically distinguishable from zero at conventional levels.** Its coefficient is 1.483, its HC3 standard error is 1.904, and its $p$-value is 0.436, so the earlier claim of statistical significance was incorrect. With only four zero-discount bundles, including the unusually popular BioShock Triple Pack, the estimate is also imprecise and potentially sensitive to individual observations. This is absence of evidence for a conditional difference, not evidence that the effect is exactly zero.

The next cell saves the complete coefficient, HC3 standard-error, 95% confidence-interval, $p$-value, sample-size, and $R^2$ table for all three declared proxy specifications to `outputs/tables/05_descriptive_regressions.csv`.


In [5]:
specs = {
    'primary_mincost_attributed': {
        'proxy_label': 'mincost_attributed (04.ipynb, primary)',
        'dependent_variable': 'log_demand',
    },
    'raw_users_own_all': {
        'proxy_label': 'users_own_all (03.ipynb raw)',
        'dependent_variable': 'log_own_all',
    },
    'reach_users_own_any': {
        'proxy_label': 'users_own_any (reach)',
        'dependent_variable': 'log_own_any',
    },
}
rhs = 'implied_discount_rate + n_bundle_items + log_price + is_zero_discount'
summary_rows = []
coefficient_rows = []
fitted_models = {}
for specification, spec in specs.items():
    dep = spec['dependent_variable']
    m = smf.ols(f'{dep} ~ {rhs}', data=d).fit(cov_type='HC3')
    fitted_models[specification] = m
    ci = m.conf_int(alpha=0.05)
    for term in m.params.index:
        coefficient_rows.append({
            'specification': specification,
            'dependent_variable': dep,
            'proxy_label': spec['proxy_label'],
            'term': term,
            'coefficient': float(m.params[term]),
            'hc3_std_error': float(m.bse[term]),
            'ci_95_lower': float(ci.loc[term, 0]),
            'ci_95_upper': float(ci.loc[term, 1]),
            'p_value': float(m.pvalues[term]),
            'nobs': int(m.nobs),
            'r_squared': float(m.rsquared),
        })
    summary_rows.append({
        'proxy': spec['proxy_label'],
        'discount_coef': m.params['implied_discount_rate'],
        'discount_hc3_z': m.tvalues['implied_discount_rate'],
        'zero_disc_coef': m.params['is_zero_discount'],
        'R2': m.rsquared,
        'n': int(m.nobs),
    })

assert np.allclose(fitted_models['primary_mincost_attributed'].params, model.params)
regression_table = pd.DataFrame(coefficient_rows)
assert len(regression_table) == len(specs) * 5
assert set(regression_table['nobs']) == {238}
regression_path = '../outputs/tables/05_descriptive_regressions.csv'
regression_table.to_csv(regression_path, index=False)

panel = pd.DataFrame(summary_rows)
print(panel.round(3).to_string(index=False))
print(f'\nsaved complete HC3 coefficient table: {regression_path} ({len(regression_table)} rows)')

                                 proxy  discount_coef  discount_hc3_z  zero_disc_coef    R2   n
mincost_attributed (04.ipynb, primary)          2.291           3.779           1.483 0.079 238
          users_own_all (03.ipynb raw)          2.424           4.160           2.403 0.090 238
                 users_own_any (reach)          2.368           3.287           1.822 0.207 238

saved complete HC3 coefficient table: ../outputs/tables/05_descriptive_regressions.csv (15 rows)


## Section 4: Why a reduced-form discount response cannot price the bundle

Initially, I looked to take the discount coefficient `beta`, assumed a constant-semi-elasticity demand `q(delta) = q_obs * exp(b * (delta - delta_obs))`, and derived a closed-form revenue-maximising discount `delta* = 1 - 1/b` (`dy/dx = 0`). Using `b = 2.291`, `delta* = 0.564`, or a 56.4% discount for every bundle.

That sketch is retired as a pricing method for three reasons:

1. **It is non-causal.** The discount coefficient is cross-sectional and confounded. Publishers set discounts strategically, so the coefficient does not recover a causal demand curve.
2. **It uses an ownership proxy, not bundle transactions.** Owning all games does not establish how they were acquired, which offers were seen, or which offers were rejected. A price-times-proxy calculation is not identified revenue.
3. **It forces one answer on every bundle.** A single semi-elasticity gives the same discount for all bundles. That uniform result is a property of the assumed functional form, not a recovered item-set-specific response.

The live bridge therefore separates two layers. Layer 1 estimates latent ranking scores from sparse implicit feedback; those scores are not willingness to pay, purchase probabilities, or structural demand parameters. Layer 2 applies predeclared pseudo-utility transformations and evaluates a stated normalized decision model. The resulting objectives are conditional within-model quantities, not dollars or estimates of Steam revenue.


## Section 5: From latent preference scores to CP-anchored fixed-bundle design

The live project uses two deliberately separate layers. The descriptive regression above remains an EDA result; it neither estimates customer demand nor calibrates the optimization layer. Layer 1 estimates latent ranking scores from implicit feedback. Layer 2 maps those scores through frozen pseudo-utility scenarios and evaluates a fixed seller-curated bundle under an explicitly assumed menu.

Two framing points follow from the documented July 2026 mechanism pivot:

- The retired cross-moment bundle-size model lets a customer choose an arbitrary set of a priced size. That does not represent a fixed curated bundle. The primary model is therefore CP-anchored Single Bundle with All, written $\mathrm{SBA}^{CP}$: component products remain available at component-pricing-optimal normalized prices, and the seller adds at most one fixed bundle.
- Matrix factorization outputs are latent preference scores. They are not valuations, willingness to pay, utilities, or purchase probabilities. Every $v_{ui}^m=T_m(s_{ui})$ below is a pseudo-utility in a declared modeling scenario $m$, not an identified economic quantity. Prices, costs, and objectives are in the same normalized scenario units, not dollars or estimates of Steam revenue.

**1. Preference estimation (Layer 1).** Construct a sparse implicit-feedback user-game matrix and complete the required model ladder before optional extensions: popularity, implicit ALS, identity-only pairwise low-rank matrix factorization, and the same matrix-factorization model with a controlled genre-metadata extension. Compare them with leakage-safe held-out ownership reconstruction, full-catalogue ranking, multiple seeds, explicit score-tie handling, paired uncertainty, and pseudo-cold-item analysis. Genre is an ablation whose incremental value must be measured; it is not presumed to help and does not receive a causal interpretation.

**2. Dependence as a descriptive bridge.** Dependence among raw behavioral signals, latent scores, and transformed pseudo-utilities must be reported separately and against frozen matched controls. Pearson correlation and additive dispersion are generally changed by nonlinear monotone transformations. Rank correlations have only their usual narrower invariance and are not preserved by arbitrary user-specific transformations that reorder users within an item. Consequently, there is no blanket claim that a correlation result survives every monotone score transform. Classic bundling theory motivates hypotheses under its stated economic assumptions; it does not determine the sign or decision relevance of dependence in this score-derived setting.

**3. Primary design experiment (Layer 2).** For each frozen pseudo-utility scenario, first estimate component-pricing-optimal normalized prices $p_i^{CP}$ on design users and hold them fixed. Then choose a feasible nonempty composition $B$, with $|B|\le C$, and normalized bundle price $b$. Component alternatives remain available. Under the primary bundle-preferred weak-tie rule, user $u$ chooses the bundle exactly when

$$
w_u(B):=\sum_{i\in B}\min\{v_{ui}^m,p_i^{CP}\}\ge b.
$$

The truncation is essential: $\sum_{i\in B}v_{ui}^m\ge b$ is the SBR rule and is not the SBA choice rule. If $c_i$ is an assumed normalized pseudo-cost, define the CP margin displaced when user $u$ switches to the bundle by

$$
A_u(B):=\sum_{i\in B}(p_i^{CP}-c_i)\mathbf 1\{v_{ui}^m\ge p_i^{CP}\}.
$$

Writing $y_u(B,b)=\mathbf 1\{w_u(B)\ge b\}$, the incremental sample-average objective is

$$
\widehat\Pi_{SBA}(B,b)=\widehat\Pi_{CP}+\frac{1}{U}\sum_u y_u(B,b)\left[b-c(B)-A_u(B)\right].
$$

Thus a bundle payment must be evaluated net of the component margin it displaces. The empty-bundle option equals CP, so optimized $\mathrm{SBA}^{CP}$ weakly dominates CP on its design objective. SBA does not nest pure bundling because the component options remain available. Fixed-composition pricing is exact at complete observed threshold blocks under the weak-tie rule; exhaustive feasible-composition search certifies only instances for which every composition is enumerated and priced exactly. Larger-pool heuristic outputs are the best solutions found unless a valid certificate or bound is available.

**4. SBR is a distinct theoretical benchmark.** Under Single Bundle with the Rest, items in $B$ are unavailable separately, so its customer-choice and nesting relationships differ from SBA. Results in *Partition and Prosper* concerning SBR hardness, tractability, half-purchase probability, covariance comparative statics, nesting, or approximation are SBR results only. They are not transferred to $\mathrm{SBA}^{CP}$ without an independent proof. The static Steam snapshot supports, at most, an SBA-like institutional description; it does not establish Steam's full historical mechanism or bundling authority for a proposed candidate pool.

**5. Decision evaluation.** Exact composition search on certifiable instances and a locked exact-suite comparison for any heuristic precede real-pool use. Complete policies (including transformation, component prices, pool, capacity, composition, bundle price, costs, and tie rule) are frozen on design users and then evaluated on assessment users without reoptimization. These are held-out comparisons of decisions conditional on pseudo-utility scenarios, not predictions of future purchases or validation against observed bundle transactions.
